# File Allocation Table (FAT)
To write files to a drive in a computer, you should first put that drive a data structure called __file system__.
Initializing a file system on a drive i.e. putting the file system structure is called __formatting__.
Then the drive is said to be __formatted (with the file system)__.

There are several file systems including FAT12, FAT16, FAT32, ExtFAT, NTFS, HPFS, ext2, ext3 etc.
The most simplest are FAT family of file systems (FAT12, FAT16, FAT32).

On FAT file systems the files are written to drive as a **linked-list** of chunks. A __chunk__ is a part the file.

## FAT12
In FAT12, four data structures are used:
- Boot Sector
- FAT Table
- Directory
- Linked List

Basically, to read a file on a drive formatted with FAT12:
1. read the **boot sector** (first 512 bytes) of the drive.
Then according to the information on the first sector:
2. read the **FAT** table.
Calculate the position of the root directory using the FAT table. With this information
3. read the **root directory**.
    - find the position of the first chunk _(head of the linked-list)_ of the file in the root directory:
4. read the first chunk of the file.
    - after finish read the position of the next chunk of the file.
    - repeat these until there is no more chunks.

## Testing

Accessing a real drive data on modern operating systems requires elevated permissions.
Instead, we will use a file with a FAT file system structure on it.
In attachment you will find an image of a floppy drive formatted with FAT12.
Open that file in binary mode as follows:

In [10]:
image = open('sample.img','rb')

## Parsing The Boot Sector

Read the boot sector (first sector) by:

In [11]:
bs = image.read(512)

You can use `struct` module of python to parse the structures.
To use that:

In [12]:
import struct

Boot sector structure is parsed by the following:

In [13]:
(
    BS_OEMName,
    BPB_BytsPerSec,
    BPB_SecPerClus,
    BPB_RsvdSecCnt,
    BPB_NumFATs,
    BPB_RootEntCnt,
    BPB_TotSec16,
    BPB_Media,
    BPB_FATSz16,
    BPB_SecPerTrk,
    BPB_NumHeads,
    BPB_HiddSec,
    BPB_TotSec32,
    BS_DrvNum,
    BS_BootSig,
    BS_VolID,
    BS_VolLab,
    BS_FilSysType,
) = struct.unpack("<3x8s HBHBHHBHHHII B1xBI 11s8s", bs[:62])

In [14]:
print(
    f"BS_OEMName:\t{BS_OEMName.decode()}",
    f"BPB_BytsPerSec:\t{BPB_BytsPerSec}",
    f"BPB_SecPerClus:\t{BPB_SecPerClus}",
    f"BPB_RsvdSecCnt:\t{BPB_RsvdSecCnt}",
    f"BPB_NumFATs:\t{BPB_NumFATs}",
    f"BPB_RootEntCnt:\t{BPB_RootEntCnt}",
    f"BPB_TotSec16:\t{BPB_TotSec16}",
    f"BPB_Media:\t{BPB_Media}",
    f"BPB_FATSz16:\t{BPB_FATSz16}",
    f"BPB_SecPerTrk:\t{BPB_SecPerTrk}",
    f"BPB_NumHeads:\t{BPB_NumHeads}",
    f"BPB_HiddSec:\t{BPB_HiddSec}",
    f"BPB_TotSec32:\t{BPB_TotSec32}",
    f"BS_DrvNum:\t{BS_DrvNum}",
    f"BS_BootSig:\t{BS_BootSig}",
    f"BS_VolID:\t{BS_VolID}",
    f"BS_VolLab:\t{BS_VolLab.decode()}",
    f"BS_FilSysType:\t{BS_FilSysType.decode()}",
    sep="\n",
)

BS_OEMName:	WINIMAGE
BPB_BytsPerSec:	512
BPB_SecPerClus:	1
BPB_RsvdSecCnt:	1
BPB_NumFATs:	2
BPB_RootEntCnt:	224
BPB_TotSec16:	2880
BPB_Media:	240
BPB_FATSz16:	9
BPB_SecPerTrk:	18
BPB_NumHeads:	2
BPB_HiddSec:	0
BPB_TotSec32:	0
BS_DrvNum:	0
BS_BootSig:	41
BS_VolID:	1100881118
BS_VolLab:	PYTHON     
BS_FilSysType:	FAT12   


In [15]:
# 1.Position Calculations

import math

# Root Directory Byte Size: 32 bytes * Maximum entry count (224)
Root_Dir_Bytes = BPB_RootEntCnt * 32 

# Root Directory Sector Count: (224 * 32) / 512 = 14 sectors
Root_Dir_Secs = math.ceil(Root_Dir_Bytes / BPB_BytsPerSec)

# FAT Start Sector: Equal to the Reserved Sector Count (1)
FAT_Start_Sec = BPB_RsvdSecCnt

# Root Directory Start Sector: Reserved (1) + (Number of FAT Copies (2) * FAT Sector Size (9)) = 19
Root_Dir_Start_Sec = BPB_RsvdSecCnt + (BPB_NumFATs * BPB_FATSz16)

# Root Directory Start Byte: 19 Sectors * 512 Bytes/Sector = 9728
Root_Dir_Start_Byte = Root_Dir_Start_Sec * BPB_BytsPerSec

# Data Region Start Byte (where Cluster 2 begins): After the Root Directory ends
Data_Start_Byte = Root_Dir_Start_Byte + (Root_Dir_Secs * BPB_BytsPerSec)
Bytes_Per_Cluster = BPB_SecPerClus * BPB_BytsPerSec # Bytes Per Cluster: 1 * 512 = 512

print(f"Root Dir Start Byte:\t{Root_Dir_Start_Byte}")
print(f"Data Start Byte:\t{Data_Start_Byte}")
print(f"Bytes Per Cluster:\t{Bytes_Per_Cluster}")


Root Dir Start Byte:	9728
Data Start Byte:	16896
Bytes Per Cluster:	512


In [16]:
#2. FAT Table

# The total size of the FAT (9 Sectors * 512 Bytes/Sector)
FAT_Size_Bytes = BPB_FATSz16 * BPB_BytsPerSec

# Move the file pointer to the start of the FAT (Sector 1)
image.seek(FAT_Start_Sec * BPB_BytsPerSec)

# Read the FAT
FAT = image.read(FAT_Size_Bytes)

print(f"FAT Size Bytes:\t{FAT_Size_Bytes}")
print("FAT table successfully read.")

FAT Size Bytes:	4608
FAT table successfully read.


In [17]:
# 3. Read the Root Directory
image.seek(Root_Dir_Start_Byte)
Root_Dir = image.read(Root_Dir_Secs * BPB_BytsPerSec)

# 4. Root Directory Scan and File Entry Lookup
target_file_name = "ASSIGN~0TXT" # This file name was determined after the initial attempt with "ASSIGNMENT.TXT" failed due to the 8.3 naming rule.
file_entry = None
LFN_ATTRIBUTE = 0x0F 

print("Root Directory Scan Proof")

# Iterate through the Root Directory data, stepping by 32 bytes (the size of one directory entry)
for i in range(0, len(Root_Dir), 32):
    entry = Root_Dir[i:i + 32]
    
    # Check if the entry is empty (0x00) or deleted (0xE5)
    if entry[0] == 0x00: break # Stop if empty, as all following entries will also be empty
    if entry[0] == 0xE5: continue # Skip if deleted
    
    attribute = entry[11]
    # Check for Long File Name (LFN) attribute (0x0F) and skip if found
    if attribute == LFN_ATTRIBUTE: continue
        
    try:
        # Decode the first 11 bytes (8.3 SFN) and strip surrounding whitespace
        file_name_stripped = entry[0:11].decode('ascii').strip()
        
        # PROOF OUTPUT
        print(f"Read Directory Entry: '{file_name_stripped}'")
        
    except UnicodeDecodeError: continue

    # Check for a match with the target SFN
    if file_name_stripped == target_file_name:
        # Extract the Start Cluster number (2 bytes, Little-endian 'H')
        start_cluster = struct.unpack('<H', entry[26:28])[0]
        # Extract the File Size (4 bytes, Little-endian 'I')
        file_size = struct.unpack('<I', entry[28:32])[0]
        
        file_entry = {"name": file_name_stripped, "start_cluster": start_cluster, "size": file_size}
        print(f"\n {file_entry['name']} abbreviated name found.")
        print(f"Start Cluster: {start_cluster}")
        break # Stop scanning once the file is found

if not file_entry: print("ERROR: File not found.")

Root Directory Scan Proof
Read Directory Entry: 'LISTS-~1IPY'
Read Directory Entry: 'WHILE-~1IPY'
Read Directory Entry: 'FLOATI~1IPY'
Read Directory Entry: 'VARIAB~1IPY'
Read Directory Entry: 'FOR-LO~1IPY'
Read Directory Entry: 'INTEGE~1IPY'
Read Directory Entry: 'CONDIT~1IPY'
Read Directory Entry: 'TUPLES~1IPY'
Read Directory Entry: 'STRING~1IPY'
Read Directory Entry: 'PYTHON'
Read Directory Entry: 'ASSIGN~0TXT'

 ASSIGN~0TXT abbreviated name found.
Start Cluster: 377


In [18]:
# 5. FAT12 Cluster Chain and File Reading

if file_entry:
    
    # The simplest function to read the 12-bit value from FAT12
    def get_fat12_entry(cluster_num, fat_data):
        # Calculate the starting byte index in the FAT (cluster_num * 1.5)
        fat_index = cluster_num * 3 // 2 
        
        if fat_index + 1 < len(fat_data):
            # Read the relevant 2 bytes (16 bits) in Little-endian format
            two_bytes = struct.unpack('<H', fat_data[fat_index:fat_index + 2])[0]
            
            # If cluster number is even, take the low 12 bits (mask with 0x0FFF)
            # If cluster number is odd, take the high 12 bits (shift right by 4)
            return (two_bytes & 0x0FFF) if cluster_num % 2 == 0 else (two_bytes >> 4)
        return 0xFFF # Default to EOF if out of bounds
        
    # Function to convert the cluster number to the disk byte offset
    def cluster_to_byte_offset(cluster_num):
        # Cluster 2 starts at Data_Start_Byte
        return Data_Start_Byte + (cluster_num - 2) * Bytes_Per_Cluster

    file_content = b""
    current_cluster = file_entry['start_cluster']
    
    print("Single Chunk and End-of-File (EOF) PROOF")
    
    # The loop should execute only once for a single-chunk file.
    # 0xFF8 is the start of the reserved EOF range for FAT12.
    while current_cluster < 0xFF8: 
        
        # PROOF 1: Show which cluster is being read
        print(f"1. STEP: Current Cluster Read: {current_cluster} (Single Chunk)")
        
        # Seek and read the cluster content
        image.seek(cluster_to_byte_offset(current_cluster))
        file_content += image.read(Bytes_Per_Cluster)
        
        # PROOF 2: Get the next value from FAT (it should be EOF)
        next_cluster = get_fat12_entry(current_cluster, FAT)
        print(f"2. STEP: Next Value Read from FAT: 0x{next_cluster:X} (Last Note/EOF)")
        
        # Move to the next cluster in the chain
        current_cluster = next_cluster
        
    # Final result: Trim the content to the actual file size
    final_content = file_content[:file_entry['size']]
    
    print("\nASSIGNMENT.TXT CONTENT")
    print(final_content.decode('ascii'))
    
    image.close()

Single Chunk and End-of-File (EOF) PROOF
1. STEP: Current Cluster Read: 377 (Single Chunk)
2. STEP: Next Value Read from FAT: 0xFFF (Last Note/EOF)

ASSIGNMENT.TXT CONTENT
Hello, World


# Assignment:
There is a file `ASSIGNMENT.TXT` in the root directory of file system image `sample.img`.
Read that file by implementing the remaining of this notebook using the Python programming language.
Add nessesary explanations as markdown. 